# 提示工程

在本书的前几章中，我们初步探索了LLM的世界。我们深入研究了各种应用，如监督分类和无监督分类；使用了专注于文本表示的模型，如BERT及其衍生模型。

随着学习的深入，我们使用了主要为文本生成而训练的模型，这类模型通常被称为生成式预训练Transformer(GPT)。它们具有强大的能力，可以根据用户的提示词生成文本。通过提示工程，我们可以设计更高效的提示词，从而提高生成文本的质量。

在本章中，我们将更详细地探索这些生成模型，并深入研究提示工程、基于生成模型的推理、输出的验证和评估。

## 6.1 使用文本生成模型

在开始学习提示工程的基础知识之前，了解文本生成模型的基础知识至关重要。我们如何选择模型？是使用专有模型还是开源模型？如何控制生成的输出？这些问题将作为我们使用文本生成模型的起点。

### 6.1.1 选择文本生成模型

选择文本生成模型首先需要在专有模型和开源模型之间做出选择。虽然专有模型通常性能更好，但在本书中我们更多关注开源模型，因为它们提供了更强的灵活性，且可以免费使用。

图6-1展示了一些具有影响力的基础模型，这些LLM在海量文本数据上进行了预训练，通常会针对特定应用进行微调。

<div align="center">
    <img src="./picture/6-1.jpg">
    <p>图6-1：基础模型通常以多种参数规模发布</p>
</div>

这些基础模型经过微调，已经产生了数百甚至数千个模型，每种模型在特定任务上的表现各有千秋。选择使用哪个模型可能是一项艰巨的任务。

我们建议从小型基础模型开始。因此，我们继续使用Phi-3-mini，它有38亿个参数，可以在显存容量不超过8 GB的设备上运行。总的来说，从小模型逐步扩展到大模型的学习体验通常比反之更顺畅。较小的模型不仅是入门的绝佳选择，还为后续进阶到更大的模型奠定了坚实的基础。

### 6.1.2 加载文本生成模型

正如前几章所述，加载模型最直接的方法是使用transformers库：

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# 加载模型和分词器
model = AutoModelForCausalLM.from_pretrained(
    "../models/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
)
tokenizer = AutoTokenizer.from_pretrained("../models/Phi-3-mini-4k-instruct")
# 创建流水线
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


与前几章相比，本章将更深入地探讨提示词模板的开发和使用。

为了说明这一点，让我们回顾一下第1章的例子，当时我们要求LLM讲一个关于鸡的笑话：

In [2]:
# 提示词
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
# 生成输出
output = pipe(messages)
print(output[0]["generated_text"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Why did the chicken join the band? Because it had the drumsticks!


在底层，transformers.pipeline首先将我们的消息转换为特定的提示词模板。我们可以通过访问底层分词器来探索这个过程：

In [3]:
# 应用提示词模板
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

<|user|>
Create a funny joke about chickens.<|end|>
<|endoftext|>


你可能还记得我们在第2章学习的特殊词元<|user|>和<|assistant|>。这个提示词模板（如图6-2所示）是模型训练期间使用的。它不仅提供了对话中“谁说了什么”的信息，还可以通过<|end|>词元指示模型应该在何时停止生成文本。这个提示词会直接传递给LLM并一次性处理完毕。

<div align="center">
    <img src="./picture/6-2.jpg">
    <p>图6-2：Phi-3与模型交互所需的模板</p>
</div>

在下一章中，我们将自定义这个模板的某些部分。在本章中，我们可以使用transformers.pipeline来帮助我们处理聊天模板。接下来，让我们探索如何控制模型的输出。

### 6.1.3 控制模型输出

除了提示工程，我们还可以通过调整模型参数来控制输出类型。下面我们来介绍一下最常用的模型参数：temperature和top_p。

模型参数控制着输出的随机性。LLM令人兴奋的技术特点之一是，它能够为完全相同的提示词生成不同的响应。每当LLM需要生成一个词元时，它都会为每个可能的词元分配一个可能性分数。

如图6-3所示，在句子“I am driving a...”​（我正在开……）中，a后面接词元car（车）或truck（卡车）的可能性通常比接词元elephant（大象）要大。然而，生成elephant的可能性仍然存在，只是要小得多。

<div align="center">
    <img src="./picture/6-3.jpg">
    <p>图6-3：模型根据可能性分数选择下一个要生成的词元</p>
</div>

在加载模型时，我们特意设置了do_sample=False，以确保输出具有一定的一致性。这意味着模型不会进行采样，只会选择最可能的下一个词元。然而，要使用temperature和top_p参数，我们需要设置do_sample=True。

temperature

temperature（温度）决定生成文本的随机性或创造性。它定义了选择本来不太可能出现的词元的概率。其基本原理是，temperature为0时每次都会生成相同的响应，因为它总是选择可能性最大的词。如图6-4所示，较高的temperature值允许生成可能性更小的词。

<div align="center">
    <img src="./picture/6-4.jpg">
    <p>图6-4：temperature越高，生成可能性更小词元的概率越高，反之亦然</p>
</div>

因此，较高的temperature（如0.8）通常会产生更多样化的输出，而较低的temperature（如0.2）会产生更具确定性的输出。

你可以在流水线中这样使用temperature：

In [4]:
# 使用较高的温度
output = pipe(messages, do_sample=True, temperature=1)
print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why don't chickens use computers? Because they're afraid of backyard hackers!


注意，每次重新运行这段代码时，输出都会改变。temperature引入了随机行为，因为模型现在会随机选择词元。

top_p

top-p采样，也称为核采样(nucleus sampling)，是一种控制LLM可以考虑哪些词元子集（核）的采样技术。它会考虑概率最高的若干词元，直到达到其累积概率限制。如果我们将top_p设置为0.1，模型会从概率最高的词元开始考虑，直到这些词元的累积概率达到0.1。如果我们将top_p设置为1，模型会考虑所有词元。

如图6-5所示，降低top_p值会减少LLM考虑的词元数量，通常会生成“更具确定性”的输出；而提高top_p值则允许LLM从更多词元中进行选择，从而生成“更具创造性”的输出。

<div align="center">
    <img src="./picture/6-5.jpg">
    <p>图6-5：较高的top_p会增加生成词元的候选集合中的词元数量，反之亦然</p>
</div>

同样，top_k参数精确控制LLM可以考虑的词元数量。如果你将其值更改为100，LLM将只考虑可能性最大的前100个词元。

你可以在流水线中这样使用top_p：

In [5]:
# 使用较高的top_p
output = pipe(messages, do_sample=True, top_p=1)
print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Why don't chickens play poker in the dark? Because they can't handle the hatch!


如表6-1所示，这些参数允许用户在创造性（高temperature和高top_p）和可预测性（低temperature和低top_p）之间进行调节。

表6-1：选择temperature和top_p值的用例

## 6.2 提示工程简介

在使用文本生成类LLM的过程中，提示工程是一个至关重要的部分。通过精心设计提示词，我们可以引导LLM生成所需的响应。无论提示词是问题、陈述还是指令，提示工程的主要目标都是引导模型生成有用的回复。

提示工程不仅仅是设计效果良好的提示词这么简单。它可以用作评估模型输出的工具，也可用于设计保障措施和风险控制方法。提示词优化的迭代过程需要不断实验。目前没有，未来也不太可能有完美的提示词设计。

在本节中，我们将介绍提示工程的常用方法，以及一些小技巧，有助于理解某些提示词的效果。这些技能让我们能够理解LLM的能力，这是与这类模型交互的基础。

我们首先回答一个问题：提示词中应该包含什么？

### 6.2.1 提示词的基本要素


LLM是一个预测引擎。基于某个输入（即提示词）​，它试图预测可能跟随其后的词。从本质上讲，只需少数几个词，就能激发LLM做出响应（见图6-6）​。

然而，尽管这个基本示例看起来可以工作，但它无法完成指定的任务。我们通常向LLM提出特定问题或任务，从而实现提示工程。为了获得所需的响应，我们需要一个结构更完善的提示词。

<div align="center">
    <img src="./picture/6-6.jpg">
    <p>图6-6：一个基本的提示词示例。由于没有给出指令，LLM将简单地尝试补全这个句子</p>
</div>

例如，如图6-7所示，我们可以让LLM将一个句子分类为正面或负面情感。我们扩展最基本的提示词，使其包含两个组件——指令本身和与指令相关的数据。

<div align="center">
    <img src="./picture/6-7.jpg">
    <p>图6-7：基本指令提示词的两个组件：指令本身及其相关的数据</p>
</div>

更复杂的用例可能需要在提示词中包含更多组件。例如，为了确保模型只输出negative（负面）或positive（正面）​，我们可以用输出指示器来引导模型。在图6-8中，我们在句子前加上“Text:”​（文本：​）​，并添加“Sentiment:”​（情感：​）​，以防止模型生成完整的句子。这种结构表明我们期望输出为negative或positive。尽管模型可能没有直接在这些组件上训练过，但它接收了足够多的指令，能够泛化到这种结构。

我们可以继续添加或更新提示词的组件，直到获得我们想要的响应。我们可以添加更多示例、更详细地描述用例、提供额外的上下文等。上述组件仅仅是示例，并未展示所有的可能。设计这些组件，关键要靠创造力。

<div align="center">
    <img src="./picture/6-8.jpg">
    <p>图6-8：扩展提示词，添加输出指示器以获得特定输出</p>
</div>

虽然提示词是单个文本片段，但将其分解为一个更大的系统的组件，会很有帮助。例如，我是否描述了问题的上下文？提示词中是否包含输出示例？

### 6.2.2 基于指令的提示词

虽然提示词有多种形式，可以是与LLM讨论哲学，也可以是与你最喜欢的超级英雄玩角色扮演，但提示词通常用于让LLM回答特定问题或解决特定任务。这称为基于指令的提示词。

图6-9展示了基于指令的提示词发挥重要作用的一些用例。我们在前面的示例中已经展示了其中之一，即监督分类。

<div align="center">
    <img src="./picture/6-9.jpg">
    <p>图6-9：基于指令的提示词的应用场景</p>
</div>

这些任务需要不同的提示词格式，更具体地说，需要向LLM提出不同的问题。这样，让LLM提炼一段文本的摘要时，就不会突然得到文本分类的结果。图6-10展示了一些应用场景下的提示词示例。

<div align="center">
    <img src="./picture/6-10.jpg">
    <p>图6-10：常见应用场景的提示词示例。注意在同一个应用场景中，指令的结构和位置可以改变</p>
</div>

虽然这些任务需要不同的指令，但用于提高输出质量的提示词优化技术实际上有很多共通之处。这些技术包括但不限于：

具体性

准确描述你想要达到的目标。让LLM“为产品写一段描述”​，不如让它“用不超过两句话撰写一段正式风格的产品描述”​。

幻觉

LLM可能会自信地生成错误信息，这被称为幻觉(hallucination)。为了降低其影响，我们可以要求LLM只在知道答案时才生成答案。如果不知道答案，可以回答“我不知道”​。

顺序

在提示词的开头或结尾放置指令。特别是对于长提示词，中间的信息往往会被遗忘。LLM往往会关注提示词的开头部分（首位效应）或结尾部分（近因效应）​。

其中，具体性可以说是最重要的。通过限制和明确模型应该生成的内容，可以减小生成与使用场景无关内容的可能性。例如，如果我们不添加“in two to three sentences”这个指令，它可能会生成完整的段落。就像人类对话一样，如果没有具体的指令或附加的上下文，模型很难判断实际任务是什么。

## 6.3 高级提示工程

从表面上看，创建一个好的提示词似乎很简单，只需要提出一个具体的问题，做到准确，添加一些示例就可以了。然而，提示工程实际上可能会很快变得很复杂，在使用LLM时，这个环节常常被低估。

在这里，我们将介绍几种构建提示词的高级技术，从构建复杂提示词的迭代工作流开始，到使用一连串的LLM获得更好的结果。最终，我们甚至会讲到高级推理技术。

### 6.3.1 提示词的潜在复杂性

正如我们在6.2节中探讨的那样，提示词通常由多个组件组成。在我们的第一个示例中，提示词包含了指令、数据和输出指示器。如前所述，提示词并不限于这三个组件，你可以根据需要构建任意复杂的提示词。

这些高级组件可以很快让提示词变得相当复杂。一些常见的组件如下。

角色定位

描述LLM应该扮演什么角色。例如，如果你想问一个关于天体物理学的问题，可以使用“你是一位天体物理学专家”​。

指令

任务本身。指令应该尽可能具体，避免留下太大的解释空间。

上下文

描述问题或任务背景的附加信息。它回答了“为什么提出这个指令”这样的问题。

格式

LLM输出生成文本的格式。如果不指定格式，LLM会自行决定格式，这在自动化系统中会造成麻烦。

受众

生成文本的目标对象。这也描述了输出的水平。在教育目的下，使用ELI5（Explain like I'm 5，​“向5岁的孩子解释”​）通常很有帮助。

语气

LLM在生成文本中应该使用的语气。如果你要给老板写一封正式的邮件，你肯定不想使用非正式的语气。

数据

与任务本身相关的主要数据。

为了进一步说明，我们扩展之前的分类提示词，使用上述所有组件，如图6-11所示。

<div align="center">
    <img src="./picture/6-11.jpg">
    <p>图6-11：一个包含多个组件的复杂提示词示例</p>
</div>

这个复杂的提示词展示了提示词的模块化特性。我们可以自由地添加和删除组件，并判断它们对输出的影响。如图6-12所示，我们可以逐步构建提示词，探索每次更改的效果。

<div align="center">
    <img src="./picture/6-12.jpg">
    <p>图6-12：对模块化组件进行迭代是提示工程的重要步骤</p>
</div>

这些变化不仅限于简单地添加或删除组件。正如我们之前在首因效应和近因效应中看到的那样，它们的顺序也会影响LLM输出的质量。换句话说，在为你的应用场景寻找最佳提示词时，实验是至关重要的。在提示工程中，我们本质上是在进行一个迭代的实验循环。

亲自动手试一试，在复杂提示词的基础上添加、删除部分组件，观察它对所生成输出的影响。你很快就会注意到哪些组件值得保留。你可以将自己的数据添加到data变量中：

In [6]:
# 提示词的组件
persona = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction = "Summarize the key findings of the paper provided.\n"
context = "Your summary should extract the most crucial points that can help researchers quickly understand the most vital information of the paper.\n"
data_format = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone = "The tone should be professional and clear.\n"
text = "MY TEXT TO SUMMARIZE"
data = f"Text to summarize: {text}"
# 完整提示词（删除和添加片段以查看它对输出的影响）
query = persona + instruction + context + data_format + audience + tone + data

我们可以添加各种组件，包括创意性组件，比如情感刺激（例如，​“这对我的职业生涯非常重要”）​。提示工程的乐趣之一在于，你可以尽可能发挥创造力，探索哪些组件的组合最适合你的应用场景。在开发适合你自己需求的格式时，几乎没有限制。

从某种意义上说，这是在对LLM所学习的内容以及它如何响应某些提示词进行逆向工程。然而，请注意，某些提示词在不同的模型上效果不同，因为这些模型的训练数据不一样，或者它们的训练目标各异。

### 6.3.2 上下文学习：提供示例

在前面的章节中，我们尝试准确描述LLM应该做什么。虽然准确和具体的描述有助于LLM理解用例，但我们可以更进一步：与其描述任务，为什么不直接展示任务呢？

我们可以为LLM提供我们想要完成的目标任务的具体示例。这通常被称为上下文学习(in-context learning)，其中，我们为模型提供正确的示例。

如图6-13所示，根据向LLM展示的示例数量，上下文学习可以有多种形式。零样本提示不使用示例，单样本提示使用一个示例，少样本提示使用两个或更多示例。

<div align="center">
    <img src="./picture/6-13.jpg">
    <p>图6-13：上下文学习中使用示例的不同形式</p>
</div>

我们相信“一例胜千言”​。这些示例直接展示了LLM应该实现什么以及如何实现。

我们可以用一个简单的例子来说明这种方法，这个例子来自描述该方法的原始论文。提示词的目标是生成一个包含虚构词的句子。为了提高生成句子的质量，我们可以向生成模型展示一个包含虚构词的句子的示例。

为此，我们需要区分用户(user)的问题和模型(assistant)提供的答案。我们还展示了如何使用模板处理这种交互：

In [7]:
# 使用虚构词的单个句子示例
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

<|user|>
A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:<|end|>
<|assistant|>
I have a Gigamuru that my uncle gave me as a gift. I love to play it at home.<|end|>
<|user|>
To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:<|end|>
<|endoftext|>


提示词展示了区分user和assistant的必要性。如果我们不这样做，就会看起来像是在自言自语。通过这些交互，我们可以生成如下输出：

In [8]:
# 生成输出
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


During the medieval reenactment, the knight skillfully screeg the wooden target with his broadsword, impressing the onlookers with his prowess.


它正确生成了回答！

与所有提示词组件一样，单样本或少样本提示并不是提示工程的终极解决方案。我们可以将它作为组件，进一步增强我们给出的描述。通过随机采样，模型仍然可以“选择”忽略指令。

### 6.3.3 链式提示：分解问题

在前面的例子中，我们探讨了将提示词分解成模块化组件以提高LLM的性能。虽然这种方法适用于许多场景，但对于高度复杂的提示词或应用场景可能并不可行。

除了在提示词内部分解问题，我们还可以在提示词之间这样做。本质上，我们是将一个提示词的输出作为下一个提示词的输入，从而创建一个连续的交互链来解决我们的问题。

为了说明这一点，假设我们想要使用LLM基于一些产品特征创建产品名称、口号和销售宣传语。我们可以要求LLM一次性完成这些任务，但这里我们将问题分解成几个部分。

如图6-14所示，我们得到一条流水线：首先创建产品名称，然后将其与产品特征结合起来作为输入来创建口号，最后使用特征、产品名称和口号来创建销售宣传语。

<div align="center">
    <img src="./picture/6-14.jpg">
    <p>图6-14：使用产品特征描述，通过链式提示创建合适的产品名称、口号和销售宣传语</p>
</div>

链式提示技术让LLM能够在每个独立问题上投入更多时间，而不是一次性解决整个问题。让我们用一个小例子来说明这一点。首先，我们为一个聊天机器人创建名称和口号：

In [9]:
# 为产品创建名称和口号
product_prompt = [
    {"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]
print(product_description)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Name: ChatSage
Slogan: "Your AI Companion for Smart Conversations"


然后，我们可以使用生成的输出作为LLM的输入来生成销售宣传语：

In [10]:
# 基于产品名称和口号生成销售宣传语
sales_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]
outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]
print(sales_pitch)

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Introducing ChatSage, your AI Companion for Smart Conversations. With ChatSage, you'll have a personalized and intelligent assistant at your fingertips, ready to engage in meaningful dialogue, provide helpful information, and enhance your daily interactions. Experience the future of communication with ChatSage – your smart conversation partner.


虽然我们需要调用两次模型，但这样做的一个重要优势是我们可以为每次调用设置不同的参数。例如，名称和口号所需的词元数量相对较少，而宣传语可以更长。

这种方法可以用于多种场景。

响应验证

让LLM对之前生成的输出进行二次检查。

并行提示

并行创建多个提示词，最后合并结果。例如，让多个LLM并行生成多个食谱，然后使用组合结果创建购物清单。

写故事

通过将问题分解成多个组件来利用LLM写书或故事。比如，先写一个概要，再发展人物角色，接着构建叙事节奏，然后深入创作对话。

在下一章中，我们将自动化这个过程，解决简单的LLM链带来的局限性。我们将把其他技术组件串联在一起，如记忆、工具等。在此之前，接下来的一节将进一步探讨链式提示的理念，介绍更复杂的链式提示方法，如自洽性、思维链和思维树。

## 6.4 使用生成模型进行推理

在前面的章节中，我们主要关注提示词的模块化组件，通过迭代来构建它们。这些高级提示工程技术，如链式提示，被证明是实现生成模型复杂推理的第一步。

推理是人类智能的核心组成部分，经常被拿来与LLM展现的类似推理的涌现行为进行比较。我们强调“类似”是因为在撰写本书时，这些模型通常被认为是通过记忆训练数据和模式匹配展现出这种行为的。

然而，它们的输出的确可以展现出复杂的行为，尽管这可能不是“真正的”推理，但仍被称为推理能力。换句话说，我们通过提示工程与LLM合作，从而模仿推理过程，改进LLM的输出。

为了实现这种推理行为，我们先来回顾人类行为中的推理。简单来说，我们的推理方法可以分为系统1和系统2两种思维过程。

系统1思维代表一种自动的、直觉的、几乎即时的过程。它与自动生成词元而没有任何自我反思行为的生成模型有相似之处。相比之下，系统2思维是一个有意识的、缓慢的、有逻辑性的过程，类似于头脑风暴和自我反思。

如果我们能赋予生成模型模仿某种形式的自我反思的能力，我们实际上就是在模拟系统2思维的方式，这往往比系统1思维产生更深思熟虑的响应。在本节中，我们将探索几种模仿人类推理的这类思维过程的技术，目的是改进模型的输出。

6.4.1 思维链：先思考再回答

生成模型迈向复杂推理的第一个重要步骤是应用一种称为思维链的方法。思维链的目标是让生成模型先“思考”​，而不是不经任何推理就直接回答问题。

如图6-15所示，我们在提示词中提供了一些示例，展示了模型在生成响应之前应该进行的推理。这些推理过程被称为“思考”​。这对于涉及较高复杂度的任务（如数学问题）帮助很大。添加这个推理步骤使模型能够在推理过程中充分利用更多的计算资源。相比于基于几个词元直接计算出完整的答案，在推理过程中每增加一个词元都能让LLM的输出更稳定。

<div align="center">
    <img src="./picture/6-15.jpg">
    <p>图6-15：思维链提示通过推理示例，引导生成模型在回答中运用推理</p>
</div>

我们使用论文作者的例子来展示这种现象：

In [11]:
# 使用思维链回答
cot_prompt = [
    {"role": "user", "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"},
    {"role": "assistant", "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."},
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"}
]
# 生成输出
outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The cafeteria started with 23 apples. They used 20 apples to make lunch, so they had 23 - 20 = 3 apples left. After buying 6 more apples, they now have 3 + 6 = 9 apples. The answer is 9.


注意，模型不仅生成了答案，还在给出答案之前提供了解释。通过这种方式，它可以利用前面生成的知识来计算最终答案。

虽然思维链是一种增强生成模型输出的好方法，但它需要在提示词中包含一个或多个推理示例，而用户可能无法获取这些示例。我们也可以不提供示例，而是直接要求生成模型进行推理（零样本思维链）​。这种方法有多种形式，一种常见且有效的方法是使用“让我们逐步思考”(Let's think step-by-step)，如图6-16所示。

<div align="center">
    <img src="./picture/6-16.jpg">
    <p>图6-16：不使用示例的思维链提示。它用“让我们逐步思考”来引导推理</p>
</div>

用之前的例子，我们可以简单地在提示词后面加上这句话来实现类似思维链的推理：

In [12]:
# 零样本思维链
zeroshot_cot_prompt = [
    {"role": "user", "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."}
]
# 生成输出
outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Step 1: Start with the initial number of apples in the cafeteria, which is 23.

Step 2: Subtract the number of apples used to make lunch, which is 20.
23 - 20 = 3 apples remaining.

Step 3: Add the number of apples bought, which is 6.
3 + 6 = 9 apples.

So, the cafeteria now has 9 apples.


无须提供示例，我们就能再次得到相同的推理行为。这就是为什么在做计算时“展示解题过程”如此重要。通过阐述推理过程，LLM可以将先前生成的信息作为指导，最终得出答案。

虽然“让我们逐步思考”可以改善输出，但不要局限于这句话的字面表述。还存在一些替代方案，比如“深呼吸，让我们逐步思考”和“让我们一步一步解决这个问题”。

### 6.4.2 自洽性：采样输出

如果我们通过temperature和top_p等参数允许一定程度的创造性，多次使用相同的提示词，可能会得到不同的结果。此时，根据词元的随机选择，输出的质量可能会提高或降低。换句话说，全靠运气！

为了抵消这种随机性并提高生成模型的性能，研究人员引入了自洽性(self-consistency)的概念。这种方法会用相同的提示词向生成模型多次提问，并将占多数的结果作为最终答案。在此过程中，可以通过调整不同的temperature和top_p值来影响每个答案，以提高采样的多样性。

如图6-17所示，通过添加思维链提示来提升模型的推理能力，同时仅将答案用于投票过程，可以进一步改进这种方法。

<div align="center">
    <img src="./picture/6-17.jpg">
    <p>图6-17：通过对多个推理路径进行采样，我们可以使用多数投票来提取最可能的答案</p>
</div>

然而，这种方法需要多次询问同一个问题，因此，尽管可以改进生成效果，但它会变得更慢。如果输出样本的数量为n，这种方法的速度就会慢至1/n。

### 6.4.3 思维树：探索中间步骤

思维链和自洽性旨在实现更复杂的推理。通过从多个“思考过程”中采样，并使它们更加深入地思考，我们可以达到改进生成模型的输出的目标。

这些技术只是触及了当前用于模拟复杂推理的方法的表面。思维树(tree-of-thought，ToT)是对这些方法的一种改进，它可以对多个想法进行深入探索。

该方法的工作原理如下。当面对需要多个推理步骤的问题时，将其分解成多个部分通常会有所帮助。在每个步骤中，如图6-18所示，生成模型会被提示探索当前问题的不同解决方案。然后，它对最佳解决方案进行投票，并继续进行下一步。

<div align="center">
    <img src="./picture/6-18.jpg">
    <p>图6-18：利用树状结构，生成模型可以生成待评分的中间思考过程。最有希望的思考过程会被保留，而较差的会被剪枝</p>
</div>

当需要考虑多条路径，比如写故事或提出创意构想时，这种方法非常有帮助。这种方法的一个缺点是需要多次调用生成模型，这会显著降低应用程序的速度。幸运的是，已经有人成功地将思维树框架转化为一种简单的提示技术。

在这种提示技术中，我们无须多次调用生成模型，而是要求模型通过模拟多个专家之间的对话来模仿这种行为。这些专家会互相质疑，直到达成共识。以下是一个思维树提示词的示例：

In [13]:
# 零样本思维树
zeroshot_tot_prompt = [
    {"role": "user", "content": "Imagine three different experts are answering this question. All experts will write down 1 step of their thinking, then share it with the group. Then all experts will go on to the next step, etc. If any expert realizes they're wrong at any point then they leave. The question is 'The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?' Make sure to discuss the results."}
]

我们可以使用这个提示词来探索LLM如何回应复杂问题：

In [14]:
# 生成输出
outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Expert 1:
Step 1: Start with the initial number of apples, which is 23.

Expert 2:
Step 1: Subtract the number of apples used for lunch, which is 20.
Step 2: Add the number of apples bought, which is 6.

Expert 3:
Step 1: Start with the initial number of apples, which is 23.
Step 2: Subtract the number of apples used for lunch, which is 20.
Step 3: Add the number of apples bought, which is 6.

Results:
All three experts arrived at the same answer: 23 - 20 + 6 = 9 apples. Therefore, the cafeteria has 9 apples left.


我们再次得到了正确答案，不过这次是通过“专家之间的讨论”得出的。这样一场展示了提示工程创造力的“专家”对话，确实很有意思。

## 6.5 输出验证

使用生成模型构建的系统和应用最终可能会投入生产。此时，验证和控制模型的输出就变得非常重要，这样可以防止应用出现故障，并创建一个稳健的生成式AI应用。

验证输出的原因可能包括如下几种。

结构化输出

默认情况下，大多数生成模型创建的是自由形式的文本，除了自然语言定义的结构外，不遵循特定的结构。某些用例需要其输出按特定格式（如JSON）进行结构化。

输出的有效性

即使我们允许模型生成结构化输出，它仍然可以自由生成其内容。例如，当要求模型在两个选项中输出其中之一时，它不应该创造第三个选项。

伦理

一些开源生成模型没有防护机制，生成的输出没有考虑安全或伦理因素。例如，一些应用场景可能要求输出不能包含不文明用语、个人身份信息、偏见、文化刻板印象等。

准确性

许多用例要求输出符合某些标准或性能，目的是仔细检查生成的信息是否事实准确、连贯、没有幻觉。

控制生成模型的输出并非易事（就像我们探讨过的top_p和temperature等参数一样）​。我们需要改造这些模型，才能使其生成符合特定准则的输出。

通常有三种控制生成模型输出的方法。

示例

提供多个预期输出的示例。

语法

控制词元选择过程。

微调

在包含预期输出的数据上对模型进行微调。

在本节中，我们将介绍前两种方法。第三种方法（模型微调）将留到第12章，届时我们将深入探讨微调方法。

## 6.5.1 提供示例

修复输出的一个简单直接的方法是向生成模型提供预期输出的示例。正如我们之前探讨的，少样本学习是一种有助于指导生成模型输出的有效技术。这种方法也可以推广至指导输出的结构。

例如，我们希望生成模型为角色扮演游戏创建角色简介。我们先不使用任何示例：

In [15]:
# 零样本学习：不提供示例
zeroshot_prompt = [
    {"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]
# 生成输出
outputs = pipe(zeroshot_prompt)
print(outputs[0]["generated_text"])

[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```json
{
  "name": "Eldrin",
  "race": "Elf",
  "class": "Ranger",
  "level": 5,
  "attributes": {
    "strength": 12,
    "dexterity": 18,
    "constitution": 14,
    "intelligence": 10,
    "wisdom": 16,
    "charisma": 13
  },
  "skills": {
    "acrobatics": 15,
    "animal_handling": 14,
    "insight": 12,
    "perception": 16,
    "survival": 13,
    "wilderness_lore": 14
  },
  "equipment": {
    "weapon": "Longsword",
    "armor": "Leather Armor",
    "accessories": ["Wooden Staff", "Leather Bracers"]
  },
  "background": "Eldrin grew up in a secluded forest village, learning the ways of the ranger from his father. He has a deep connection with nature and animals, and is skilled in tracking and archery. He is also a skilled diplomat, able to negotiate with both humans and creatures alike."
}
```


这个被截断的输出不是有效的JSON，因为模型在开始生成charisma（魅力）属性后就停止了词元生成。此外，我们可能不想要某些特定的属性。对此，我们可以为模型提供一些示例来说明预期的格式：

In [16]:
# 单样本学习：提供输出结构示例
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:
{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [
    {"role": "user", "content": one_shot_template}
]
# 生成输出
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "description": "A cunning rogue with a mysterious past, skilled in stealth and deception.",
  "name": "Shadowcloak",
  "armor": "Leather Hood",
  "weapon": "Dagger"
}


模型完美地遵循了我们给出的示例，输出更加一致。这证明了少样本学习的重要性，它不仅可以改进输出的内容，还能改进其结构。

需要注意的是，模型输出是否遵循指定的格式仍然取决于模型本身。有些模型比其他模型更善于遵循指令。

### 6.5.2 语法：约束采样

少样本学习有一个很大的缺点：我们无法明确地阻止生成某些输出。虽然我们引导模型并给出指令，但它可能仍然不会完全遵循。

为此，一些用于约束和验证生成模型输出的软件包迅速面世，如Guidance、Guardrails和LMQL。在某种程度上，它们利用生成模型来验证自己的输出，如图6-19所示。生成模型将输出作为新的提示词，并尝试基于一些预定义的规则进行验证。

<div align="center">
    <img src="./picture/6-19.jpg">
    <p>图6-19：使用LLM检查输出是否正确遵循我们的规则</p>
</div>

同样，如图6-20所示，这种验证过程也可以用于控制输出的格式。因为我们已经知道它应该如何构建，所以可以自己生成部分格式。

<div align="center">
    <img src="./picture/6-20.jpg">
    <p>图6-20：使用LLM仅生成我们事先不知道的部分信息</p>
</div>

这个过程可以更进一步，我们不必在生成完输出后再验证，而是可以在词元采样过程中进行验证。在采样词元时，我们可以定义一些LLM在选择下一个词元时应遵循的语法或规则。例如，如果我们在提示词中要求模型在执行情感分类时仅返回positive、negative或neutral（正面、负面或中性）​，它可能仍会返回其他内容。如图6-21所示，通过约束采样过程，我们可以让LLM只输出我们感兴趣的内容。注意，这仍然会受到top_p和temperature等参数的影响。

<div align="center">
    <img src="./picture/6-21.jpg">
    <p>图6-21：将词元选择范围限制为三个：positive、neutral和negative</p>
</div>

让我们用llama-cpp-python来说明这种现象，这是一个类似transformers的库，我们可以用它来加载语言模型。它通常用于高效加载和使用压缩模型（借助量化，参见第12章）​，但我们也可以用它来应用JSON语法。

我们加载本章中一直在用的模型，但使用一个不同的格式，即GGUF。llama-cpp-python需要这种格式，它通常用于压缩（量化）模型。

由于我们要加载一个新模型，建议重启Python环境。这将清除任何先前的模型并清空内存与显存。你也可以运行以下代码来清空：

In [17]:
import gc
import torch
del model, tokenizer, pipe
# 清空内存与显存
gc.collect()
torch.cuda.empty_cache()

现在我们已经清空了内存与显存，可以加载Phi-3了。我们将n_gpu_layers设置为-1，表示我们希望模型的所有层都在GPU上运行。n_ctx指的是模型的上下文长度。repo_id和filename指向模型所在的Hugging Face仓库：

In [18]:
from llama_cpp.llama import Llama
# 加载Phi-3
llm = Llama(
    model_path="../models/Phi-3-mini-4k-instruct-q4.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False
)


要使用内部的JSON语法生成输出，我们只需要将response_format指定为一个JSON对象。在底层，它会应用JSON语法来确保输出符合该格式。

为了说明这一点，我们要求模型用JSON格式创建一个用于角色扮演游戏《龙与地下城》(Dungeons & Dragons)的角色：

In [19]:
# 生成输出
output = llm.create_chat_completion(
    messages=[
        {"role": "user", "content": "Create a warrior for an RPG in JSON for mat."},
    ],
    response_format={"type": "json_object"},
    temperature=0,
)['choices'][0]['message']["content"]

为了检查输出是否真的是JSON格式，我们可以尝试将其作为JSON处理：

In [20]:
import json
# 格式化为JSON
json_output = json.dumps(json.loads(output), indent=4)
print(json_output)

{
    "Warrior": {
        "Name": "Aldan the Brave",
        "Class": "Warrior",
        "Strength": 18,
        "Dexterity": 10,
        "Constitution": 16,
        "Intelligence": 8,
        "Wisdom": 10,
        "Charisma": 7,
        "Skills": [
            "Athletics",
            "Defense",
            "Intimidation"
        ],
        "Equipment": {
            "Weapons": [
                "Longsword",
                "Shield"
            ],
            "Armor": [
                "Chainmail Armor",
                "Leather Tunic"
            ],
            "Accessories": [
                "Warrior's Amulet",
                "Pouch of Healing Herbs"
            ]
        },
        "Abilities": {
            "Primary": [
                "Charge",
                "Shield Bash",
                "Power Attack"
            ],
            "Secondary": [
                "Shield Block",
                "Dodge Roll",
                "Heavy Strike"
            ]
        },
        "Langu

输出已正确格式化为JSON。这让我们能够更有信心地在需要输出遵循特定格式的应用中使用生成模型。

## 6.6 小结

在本章中，我们探讨了通过提示工程和输出验证使用生成模型的基础知识。我们关注了提示工程所涉及的创造性和潜在的复杂性。在不同应用场景中，不管是保证生成适当的输出，还是优化输出质量，提示词的多种组件都至关重要。

我们进一步探讨了上下文学习和思维链等高级提示工程技术。这些方法通过提供示例思考过程或鼓励逐步思考的短语，来引导生成模型解决复杂问题，从而模仿人类的推理过程。

总的来说，本章展示了提示工程对使用LLM至关重要，因为提示词使我们能够有效地向模型传达我们的需求和偏好。通过掌握提示工程技术，我们可以释放LLM的部分潜力，生成满足我们需求的高质量响应。

下一章将在这些概念的基础上，探索利用生成模型的更高级的技术。在提示工程之外，我们还将探索LLM如何使用外部记忆和工具。